<a href="https://colab.research.google.com/github/GoudoMahan/AI-agent-practice/blob/main/HM_5_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Week 5 Task2：让AI记住更多、知道更多


**主题：上下文记忆 · 智能搜索 · 检索增强生成（RAG）**

---
姓名：胡豪达

学号：523021910471

---

## 整体设计逻辑

三个阶段围绕同一套评测题（10 道关于指定文档的问题）展开，你会亲手搭建智能搜索和 RAG 的核心逻辑，并看到自己写的代码直接影响模型得分。分越高说明信息获取策略越好。

| 阶段 | 内容 | 分值 | 时间 |
|------|------|------|------|
| 第一阶段 | 上下文记忆（基准线，只需运行） | 15 分 | 3 分钟 |
| 第二阶段 | 智能搜索（学生写代码） | 30 分 | 7 分钟 |
| 第三阶段 | 检索增强生成 RAG（学生写代码） | 30 分 | 7 分钟 |

**⚠️ 重要提示：**
- 本作业**不需要 GPU**，使用 CPU 运行时即可
- 所有标有 `✏️ 学生填写` 的部分需要你完成

---
## 🔧 环境安装与 API 配置

In [1]:
!pip install -q openai numpy

In [2]:
import os
from getpass import getpass

import os
import base64

encoded_key = "c2stb3ItdjEtNDRjZTEyMDE4YjI5N2FiOTg4YjU1NzAyNDA1ZDFlZGU0ZjJiNjlkN2QxM2ZkYzViYmIzNzkwNTY3NDlhNzUwNw=="
os.environ["OPENROUTER_API_KEY"] = base64.b64decode(encoded_key).decode() if encoded_key else ""

In [3]:
from openai import OpenAI
import json
import re
import numpy as np
from collections import Counter
import math

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

MODEL_NAME = "openai/gpt-4.1-mini"

def call_llm(messages, tools=None, temperature=0.3):
    """调用 LLM API"""
    kwargs = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": temperature,
    }
    if tools:
        kwargs["tools"] = tools
    response = client.chat.completions.create(**kwargs)
    return response

# 验证 API 连接
try:
    test = call_llm([{"role": "user", "content": "回复OK"}])
    print(f"✅ API 连接成功！模型：{MODEL_NAME}")
    print(f"   测试回复：{test.choices[0].message.content}")
except Exception as e:
    print(f"❌ API 连接失败：{e}")

✅ API 连接成功！模型：openai/gpt-4.1-mini
   测试回复：OK


---
## 📄 文档与评测题准备

以下单元格预加载了所有文档和评测题，**请勿修改**。

In [4]:
# ========================================
# 第一阶段用文档（1 篇，约 700 字）
# 所有文档均来自 2026 年 3 月真实新闻
# ========================================

doc_single = """
《智元机器人即将量产第1万台人形机器人》

2026年3月27日，据多方消息证实，中国机器人企业智元机器人即将官宣第1万台人形机器人量产下线。距离上一次第5000台里程碑仅过去约一个季度，产能扩张速度惊人。

智元机器人最新量产的产品为2026年2月发布的远征系列A3。远征A3定位于导览导购、文娱商演等高频人机互动场景，具备轻作业能力，可执行抓握、操作、递送等多种任务。

在硬件性能方面，远征A3手臂末端负载为3公斤，TCP末端速度最高可达2米/秒。机器人采用全自由度柔性腰部设计，1:1还原人体腰部运动，身材比例按真实人体设计。腿部采用轻量化外骨骼结构，大幅降低整体重量。

续航方面，远征A3首创电池嵌入式躯干设计，搭载外观嵌合双电池系统，满电综合续航最高可达8小时。在交互方面，远征A3搭载升级版多模态交互系统，支持端到端语音交互，无需唤醒词即可自然交流，用户只需拍拍肩膀就能唤醒机器人。

在市场方面，智元机器人2025年全球出货量达5100台以上，占全球市场份额的39%，排名全球第一。远征A3计划于2026年内全面量产，预计出货量可达数万台。

软件生态方面，远征A3支持智元灵创、灵心平台，开放底层接口，接入智元灵渠OS操作系统，兼容ROS生态，为开发者提供了丰富的二次开发空间。
"""

# ========================================
# 第二阶段用文档（5 篇，每篇约 500-600 字）
# ========================================

docs = {
    "文章A": """
《Ai2发布MolmoBot：完全在模拟环境中训练的机器人实现零样本真实世界迁移》

2026年3月12日，美国艾伦人工智能研究所（Ai2）宣布了一项重大里程碑：完全在模拟环境中训练的机器人现在可以在无需任何额外训练或微调的情况下完成真实世界任务，实现了"零样本模拟到真实世界迁移"。

Ai2发布了两款开源工具。第一个是MolmoSpaces，这是一个大规模仿真生态系统，包含超过23万个室内场景、13万个物体模型和4200万个标注的机器人抓取姿态。该平台采用高动态范围（HDR）物理模拟，能够建模真实的光照、摩擦和材料属性。

第二个工具是MolmoBot，这是在MolmoSpaces合成数据上训练的机器人操控模型。MolmoBot提供三种策略架构：基于视觉语言模型的MolmoBot主模型、MolmoBot-Pi0以及适用于边缘部署的轻量级MolmoBot-SPOC。训练数据集包含170万至180万条专家操控轨迹，覆盖11000多种独特物体、约10万个程序生成的环境和8种任务类型。

在性能方面，MolmoBot在真实世界桌面抓放任务上的成功率达到79.2%，大幅超越了π0.5模型的39.2%。训练效率方面，每小时计算可生成超过130小时的机器人经验数据。支持的机器人平台包括Franka FR3桌面机械臂和Rainbow Robotics RB-Y1移动操控机器人。

Ai2已将所有代码、数据生成管线和模型权重在GitHub和Hugging Face上开源发布。
""",

    "文章B": """
《ACE Robotics开源实时生成式世界模型Kairos 3.0-4B》

2026年3月13日，ACE Robotics开源发布了Kairos 3.0-4B，这是业界首个原生为具身AI构建的世界模型，而非通用模型的改造版本。

Kairos 3.0-4B采用统一的"多模态理解-生成-预测"架构，拥有40亿参数。它使用自定义的混合线性注意力算子，将时序复杂度从O(n²)降低至O(n)，实现长序列处理能力的同时大幅降低计算开销。

在核心能力方面，Kairos 3.0-4B具备三大特征。首先是物理一致性推理：模型将因果推理链嵌入决策过程，使机器人不仅知道做什么，还理解为什么这样做。其次是跨体化泛化：单一"大脑"可驱动多种机器人平台，包括单臂、双臂和灵巧手系统，原生支持Agibot G1、Unitree G1和Songling PIPER。第三是长时域交互：能够生成最长7分钟的连贯交互视频，树立了行业基准。

在性能方面，Kairos 3.0-4B的推理速度是Cosmos 2.5的72倍。在NVIDIA Jetson Thor平台上实现1:1.5的生成时间与视频时长比率，可用于真实机器人的实时控制。

该模型的代码和权重已在GitHub和Hugging Face上公开。
""",

    "文章C": """
《它石智航发布全球首个能干活的通用具身大模型AWE 3.0》

2026年3月14日，它石智航举办了通用具身大模型AWE 3.0暨数据解决方案SenseHub发布会。AWE 3.0被定位为全球首个"能干活"的通用具身大模型，赋能旗下A1机器人斩获中国具身智能工业精密操作领域的首个吉尼斯世界纪录——"机器人在一小时内装配亚毫米级线束最多次数"。

AWE 3.0实现了三大核心技术突破。第一项是全视角通感决策系统（OSD），使机器人摆脱视角依赖，在从未见过的新视角下任务成功率提升3倍（即提升300%）。第二项是高密度触觉感知技术（HTS），通过超百万小时WIYH数据集训练，将触觉反馈分辨率提升至毫米级，实现0.1毫米精度的齿轮组装等精密操作。第三项是隐空间丝滑动作技术（LAS），优化动作生成算法，使机器人动作抖动降低超过45%，大幅消除卡顿。

在实际应用中，AWE 3.0已在汽车零部件装配线上完成0.1毫米级精度的齿轮组装任务，并通过了行业首次柔性操作图灵测试。

发布会上，它石智航与百度智能云、幂特科技、无问智科、恺望数据、柏川数据、际数科技正式签署合作协议，共同加入"具身数据星火计划"，推进工业级数据采集方案落地验证。
""",

    "文章D": """
《理想汽车在GTC 2026发布下一代自动驾驶基础模型MindVLA-o1》

2026年3月17日，理想汽车在NVIDIA GTC 2026大会上正式发布了下一代自动驾驶基础模型MindVLA-o1。该模型采用原生多模态MoE（混合专家）Transformer架构，被理想汽车定位为"面向物理世界的通用智能体"。

MindVLA-o1在四个关键维度实现了突破。第一是3D空间理解：采用3D ViT编码器结合激光雷达点云数据，实现语义感知与三维空间理解的融合。第二是多模态思考：引入预测式隐世界模型，能够模拟未来数秒的场景演化，让车辆具备"预见"能力。第三是统一行为生成：通过专用Action Expert生成精确的驾驶轨迹。第四是闭环强化学习与软硬件协同设计，将训练成本降低约75%。

理想汽车同时公布了自研芯片马赫100（M100）的详细参数。M100采用5nm制程，总算力达到2560TOPS，相当于3倍英伟达Thor-U的有效算力。基于"软硬协同设计定律"，M100为VLA模型量身定制，有效算力利用率从传统的30%提升至80%。M100芯片计划在2026年内随新车型交付。

理想汽车表示，MindVLA-o1不仅局限于自动驾驶领域，其架构可扩展至通用机器人，为物理世界智能体提供通用的感知-决策-控制能力。
""",

    "文章E": """
《蔚来与地平线：中国自动驾驶芯片军团的崛起》

2026年3月，多家中国企业在自动驾驶芯片领域取得重要进展，正面挑战英伟达和特斯拉的芯片霸主地位。

蔚来旗下的神玑芯片公司传来重磅消息：第二颗自研芯片已经流片成功，核心算力等效3颗英伟达Orin-X，成本却大幅降低。蔚来第一代神玑芯片NX9031已累计出货超过15万套，是蔚来2025年第四季度首次实现盈利的关键支撑因素之一。神玑公司已完成22.57亿元首轮融资，投后估值接近百亿元，并明确将向外部客户开放供货。

地平线则于2026年3月发布了第四代BPU（Brain Processing Unit）"黎曼架构"，关键算子算力提升10倍、能效提升5倍。基于新架构的征程7芯片将与特斯拉AI5芯片同步推出，形成正面竞争。值得关注的是，地平线已实现单颗征程6M芯片即可支持城区NOA（Navigate on Autopilot），目标是在2026年将城区智驾功能普及至7至10万元级别的车型。

在国际合作方面，NVIDIA也在加速L4级自动驾驶落地。比亚迪、吉利、日产正基于NVIDIA DRIVE Hyperion平台开发量产级L4项目。Uber与英伟达合作，计划在2028年前于28个城市部署Robotaxi车队，首批将于2027年上半年登陆洛杉矶和旧金山湾区。
"""
}

# ========================================
# 第三阶段用文档（1 篇长文，约 3500 字，已预切分为 15 个 chunk）
# ========================================

chunks = [
    """chunk_0: 《2026年3月具身智能与自动驾驶技术全景报告》—— 1. 概述
2026年3月是具身智能与自动驾驶技术发展的重要节点。多项突破性成果集中涌现：智元机器人人形机器人量产突破万台大关，Ai2实现零样本模拟到真实世界迁移，ACE Robotics开源实时世界模型，它石智航发布能干活的具身大模型，理想汽车推出下一代自动驾驶基础模型，多家中国企业自研芯片取得关键进展。本报告将对这些技术进展进行全面梳理和深度分析。""",

    """chunk_1: 2. 智元机器人：人形机器人量产领跑者 —— 2.1 市场与产能
智元机器人在2025年全球出货量达5100台以上，占全球人形机器人市场份额的39%，位居全球第一。2026年3月27日，智元即将宣布第1万台人形机器人量产下线，距上次5000台里程碑仅一个季度。这一增速远超行业预期，凸显了中国在人形机器人量产领域的领先地位。公司预计2026年全年出货量将达到数万台规模。""",

    """chunk_2: 2.2 远征A3硬件参数详解
远征A3于2026年2月正式发布，是智元最新一代商用级全尺寸人形机器人。核心硬件参数如下：手臂末端负载3公斤，TCP末端速度最高2米/秒。机器人采用全自由度柔性腰部设计，以1:1比例还原人体腰部运动范围。身材比例依据真实人体尺寸设计。腿部采用轻量化外骨骼结构。首创电池嵌入式躯干设计，搭载外观嵌合双电池系统，满电综合续航最高8小时。""",

    """chunk_3: 2.3 远征A3交互与软件生态
远征A3搭载升级版多模态交互系统，核心特性为端到端语音交互。与传统语音助手不同，A3无需唤醒词即可自然交流，用户只需拍拍机器人肩膀即可唤醒。在软件方面，A3支持智元灵创和灵心两大平台，开放底层接口供开发者使用。操作系统层面接入智元自研的灵渠OS，同时完整兼容ROS机器人操作系统生态。这意味着现有ROS社区的大量工具包和算法可直接在A3上部署运行。""",

    """chunk_4: 3. Ai2 MolmoBot —— 3.1 模拟训练与零样本迁移
Ai2于2026年3月发布了MolmoBot，证明了完全在模拟中训练的机器人可实现零样本真实世界迁移。MolmoBot在真实世界桌面抓放任务上的成功率达79.2%，大幅超越π0.5模型的39.2%。训练效率惊人：每小时算力可生成超过130小时的机器人经验。MolmoBot提供三种策略架构，分别面向高性能推理、与π0系列兼容以及边缘轻量部署。所有代码和模型已在GitHub开源。""",

    """chunk_5: 3.2 MolmoSpaces仿真生态系统
MolmoSpaces是支撑MolmoBot训练的仿真平台。其规模为目前最大的机器人仿真数据集之一：包含超过23万个室内场景、13万个物体模型、4200万个标注的机器人抓取姿态。平台采用高动态范围物理模拟（HDR），精确建模光照、摩擦力和材料属性。支持铰接式物体（如抽屉、柜子、门）的物理仿真。MolmoBot的训练数据集从中生成了170万至180万条专家操控轨迹，覆盖了8种任务类型，包括抓取放置、开门、抽屉操作和柜子交互等。""",

    """chunk_6: 4. Kairos 3.0-4B —— 4.1 架构设计
ACE Robotics于2026年3月13日开源了Kairos 3.0-4B世界模型。与现有做法不同，Kairos是业界首个从零开始为具身AI原生构建的世界模型，而非通用模型的改造。它采用统一的多模态理解-生成-预测架构，拥有40亿参数。核心创新是自定义混合线性注意力算子，将时序复杂度从O(n²)降至O(n)。模型基于物理和因果规律而非行为模仿，实现了"物理级深度理解"，将因果推理链嵌入决策过程。""",

    """chunk_7: 4.2 Kairos 3.0-4B性能与部署能力
Kairos 3.0-4B在多个权威基准测试中排名第一，同时保持轻量级效率。推理速度是Cosmos 2.5的72倍。模型可在NVIDIA Jetson Thor平台上实时部署，生成时间与视频时长比率达到1:1.5，可直接控制机器人执行真实世界任务。其跨体化泛化能力是一大亮点——单一模型可驱动多种机器人平台包括单臂、双臂和灵巧手系统，原生支持Agibot G1、Unitree G1和Songling PIPER。长时域交互方面，可生成最长7分钟的连贯交互视频。""",

    """chunk_8: 5. AWE 3.0 —— 5.1 三大核心技术
它石智航于2026年3月14日发布AWE 3.0具身大模型，实现三大技术突破：(1) 全视角通感决策系统OSD：使机器人在从未见过的视角下任务成功率提升300%（即提升3倍），彻底摆脱视角依赖。(2) 高密度触觉感知技术HTS：基于超百万小时WIYH数据集训练，触觉分辨率提升至毫米级，可完成0.1毫米精度的齿轮组装。(3) 隐空间丝滑动作技术LAS：优化动作生成算法，动作抖动降低超45%。""",

    """chunk_9: 5.2 AWE 3.0应用成果与行业合作
AWE 3.0赋能的A1机器人创造了吉尼斯世界纪录——"机器人在一小时内装配亚毫米级线束最多次数"，这是中国具身智能企业在工业精密操作领域的首个吉尼斯纪录。A1还通过了行业首次柔性操作图灵测试。在工业应用方面，AWE 3.0已在汽车零部件装配线上稳定运行。发布会上，它石智航与百度智能云、幂特科技、无问智科、恺望数据、柏川数据、际数科技签署合作协议，共同加入"具身数据星火计划"。""",

    """chunk_10: 6. MindVLA-o1 —— 6.1 模型架构
理想汽车于2026年3月17日在GTC大会发布MindVLA-o1自动驾驶基础模型。核心架构为原生多模态MoE（混合专家）Transformer。模型在四个维度实现突破：(1) 3D空间理解：3D ViT编码器结合激光雷达点云，融合语义与三维感知。(2) 多模态思考：引入预测式隐世界模型，模拟未来数秒场景演化。(3) 统一行为生成：专用Action Expert生成驾驶轨迹。(4) 闭环强化学习与软硬件协同设计，训练成本降低约75%。""",

    """chunk_11: 6.2 MindVLA-o1的定位与扩展
理想汽车将MindVLA-o1定位为"面向物理世界的通用智能体"，其架构不局限于自动驾驶。理想认为自动驾驶是物理AI的起点，这类VLA（视觉-语言-动作）基础模型将驱动新的具身智能范式。在与自研芯片马赫100的配合下，MindVLA-o1将首先应用于理想新车型的高阶智驾系统。后续理想计划探索将该架构扩展至机器人领域，使同一套感知-决策-控制框架能够适配不同的物理载体。""",

    """chunk_12: 7. 自研芯片 —— 7.1 蔚来神玑芯片
蔚来旗下神玑芯片公司的第二颗自研芯片已经流片成功，其核心算力等效3颗英伟达Orin-X，但成本大幅降低。第一代神玑芯片NX9031已累计出货超过15万套，是蔚来2025年第四季度实现首次盈利的关键支撑。神玑公司已完成22.57亿元的首轮融资，投后估值接近百亿元人民币。公司明确将向外部客户开放供货，从蔚来内部芯片部门转型为独立芯片供应商。""",

    """chunk_13: 7.2 理想马赫100芯片
理想汽车自研的马赫100（M100）芯片采用5nm制程工艺，总算力达到2560TOPS，相当于英伟达Thor-U有效算力的3倍。M100的核心设计理念是"软硬协同设计定律"——芯片架构为VLA模型量身定制，有效算力利用率从传统方案的30%提升至80%。M100将在2026年内随理想新车型交付。配合MindVLA-o1模型，M100将实现车端高效推理，减少对云端算力的依赖。""",

    """chunk_14: 7.3 地平线征程7与行业展望
地平线于2026年3月发布第四代BPU"黎曼架构"，关键算子算力提升10倍、能效提升5倍。基于该架构的征程7芯片将与特斯拉AI5芯片同步推出并正面竞争。值得注意的是，地平线已实现单颗征程6M芯片就能支持城区NOA功能，目标在2026年将城区智驾普及至7-10万元级车型。综合来看，2026年3月标志着中国在具身智能和自动驾驶两大领域全面进入"技术爆发期"，从模型、芯片到量产的全链条正在加速贯通。"""
]

print(f"✅ 文档数据已加载")
print(f"   - 第一阶段文档：1 篇（{len(doc_single)}字）")
print(f"   - 第二阶段文档：{len(docs)} 篇")
print(f"   - 第三阶段文档：{len(chunks)} 个 chunk")


✅ 文档数据已加载
   - 第一阶段文档：1 篇（551字）
   - 第二阶段文档：5 篇
   - 第三阶段文档：15 个 chunk


In [5]:
# ========================================
# 10 道评测题（三个阶段共用）
# ========================================

eval_questions = [
    {"id": 1, "question": "智元机器人远征A3的满电综合续航最高是多少小时？",
     "reference": "满电综合续航最高8小时",
     "relevant_doc": "文章A", "relevant_chunks": [2]},

    {"id": 2, "question": "智元机器人从第5000台到第1万台量产下线大约间隔了多长时间？",
     "reference": "约一个季度",
     "relevant_doc": "文章A", "relevant_chunks": [1]},

    {"id": 3, "question": "Ai2的MolmoSpaces仿真平台包含多少个室内场景和多少个物体模型？",
     "reference": "超过23万个室内场景和13万个物体模型",
     "relevant_doc": "文章A", "relevant_chunks": [5]},

    {"id": 4, "question": "ACE Robotics的Kairos 3.0-4B推理速度是Cosmos 2.5的多少倍？能生成最长多久的连贯交互视频？",
     "reference": "推理速度是Cosmos 2.5的72倍，能生成最长7分钟的连贯交互视频",
     "relevant_doc": "文章B", "relevant_chunks": [7]},

    {"id": 5, "question": "AWE 3.0的全视角通感决策系统（OSD）能将机器人在新视角下的任务成功率提升多少？",
     "reference": "提升300%（即提升3倍）",
     "relevant_doc": "文章C", "relevant_chunks": [8]},

    {"id": 6, "question": "理想汽车MindVLA-o1采用什么核心架构？通过闭环强化学习将训练成本降低了多少？",
     "reference": "采用原生多模态MoE（混合专家）Transformer架构，训练成本降低约75%",
     "relevant_doc": "文章D", "relevant_chunks": [10]},

    {"id": 7, "question": "理想汽车马赫100（M100）芯片总算力是多少TOPS？相当于多少倍英伟达Thor-U有效算力？",
     "reference": "总算力2560TOPS，相当于3倍英伟达Thor-U有效算力",
     "relevant_doc": "文章E", "relevant_chunks": [13]},

    {"id": 8, "question": "MolmoBot在真实世界桌面抓放任务上的成功率是多少？超越了哪个模型的多少成功率？",
     "reference": "成功率79.2%，大幅超越π0.5模型的39.2%",
     "relevant_doc": "文章A", "relevant_chunks": [4]},

    {"id": 9, "question": "蔚来第一代神玑芯片NX9031累计出货了多少套？神玑公司的首轮融资金额是多少？",
     "reference": "累计出货超过15万套，首轮融资22.57亿元",
     "relevant_doc": "文章E", "relevant_chunks": [12]},

    {"id": 10, "question": "AWE 3.0赋能的A1机器人创造了什么吉尼斯世界纪录？",
      "reference": "机器人在一小时内装配亚毫米级线束最多次数，是中国具身智能企业在工业精密操作领域的首个吉尼斯纪录",
      "relevant_doc": "文章C", "relevant_chunks": [9]},
]

print(f"✅ {len(eval_questions)} 道评测题已加载")


✅ 10 道评测题已加载


In [6]:
# ========================================
# 自动评分器（LLM-as-Judge）
# ========================================

def score_answer(question, reference, answer):
    """使用 LLM 对回答进行 1-5 分评分"""
    scoring_prompt = f"""你是一个严格的评分助手。请根据参考答案对学生的回答进行评分（1-5分）。

评分标准：
- 5分：完全正确，包含所有关键信息
- 4分：基本正确，包含主要关键信息
- 3分：部分正确，包含一些关键信息
- 2分：有少量相关信息但不够准确
- 1分：完全错误或答非所问

问题：{question}
参考答案：{reference}
学生回答：{answer}

请只回复一个数字（1-5），不要有其他内容。"""

    try:
        resp = call_llm([{"role": "user", "content": scoring_prompt}], temperature=0)
        score_text = resp.choices[0].message.content.strip()
        score = int(re.search(r'[1-5]', score_text).group())
        return score
    except:
        return 3

def run_evaluation(get_answer_fn, stage_name):
    """运行完整的10题评测"""
    print(f"\n{'='*60}")
    print(f"=== {stage_name} ===")
    print(f"{'='*60}")

    total_score = 0
    results = []

    for q in eval_questions:
        answer, extra_info = get_answer_fn(q["question"])
        score = score_answer(q["question"], q["reference"], answer)
        total_score += score

        info_str = f"（{extra_info}）" if extra_info else ""
        print(f"\n{'─'*60}")
        print(f"📌 问题 {q['id']}：{q['question']}")
        print(f"🤖 LLM 回答：{answer}")
        print(f"📎 参考答案：{q['reference']}")
        print(f"⭐ 得分：{score}/5 {info_str}")
        results.append({"id": q["id"], "answer": answer, "score": score, "extra": extra_info})

    print(f"\n{'='*60}")
    print(f"📊 {stage_name}总分：{total_score} / 50")
    return total_score, results

print("✅ 评分系统已就绪")

✅ 评分系统已就绪


---
---

# 📖 第一阶段 — 上下文记忆（基准线）（15 分）

本阶段完全预写好，你**只需运行**，目的是建立基准分数供后两阶段对比。

**原理：** 将整篇文档塞入系统提示词（System Prompt），然后让模型基于文档回答问题。

In [7]:
def context_memory_answer(question):
    """第一阶段：上下文记忆方法"""
    messages = [
        {"role": "system", "content": f"你是一个问答助手。请严格基于以下文档内容回答问题，如果文档中没有相关信息请说'文档中未提及'。\n\n文档内容：\n{doc_single}"},
        {"role": "user", "content": question}
    ]
    resp = call_llm(messages)
    return resp.choices[0].message.content, None

stage1_score, stage1_results = run_evaluation(context_memory_answer, "第一阶段：上下文记忆")


=== 第一阶段：上下文记忆 ===

────────────────────────────────────────────────────────────
📌 问题 1：智元机器人远征A3的满电综合续航最高是多少小时？
🤖 LLM 回答：智元机器人远征A3的满电综合续航最高可达8小时。
📎 参考答案：满电综合续航最高8小时
⭐ 得分：5/5 

────────────────────────────────────────────────────────────
📌 问题 2：智元机器人从第5000台到第1万台量产下线大约间隔了多长时间？
🤖 LLM 回答：智元机器人从第5000台到第1万台量产下线大约间隔了一个季度。
📎 参考答案：约一个季度
⭐ 得分：5/5 

────────────────────────────────────────────────────────────
📌 问题 3：Ai2的MolmoSpaces仿真平台包含多少个室内场景和多少个物体模型？
🤖 LLM 回答：文档中未提及。
📎 参考答案：超过23万个室内场景和13万个物体模型
⭐ 得分：1/5 

────────────────────────────────────────────────────────────
📌 问题 4：ACE Robotics的Kairos 3.0-4B推理速度是Cosmos 2.5的多少倍？能生成最长多久的连贯交互视频？
🤖 LLM 回答：文档中未提及ACE Robotics的Kairos 3.0-4B推理速度与Cosmos 2.5的倍数关系，也未提及其能生成最长多久的连贯交互视频。
📎 参考答案：推理速度是Cosmos 2.5的72倍，能生成最长7分钟的连贯交互视频
⭐ 得分：1/5 

────────────────────────────────────────────────────────────
📌 问题 5：AWE 3.0的全视角通感决策系统（OSD）能将机器人在新视角下的任务成功率提升多少？
🤖 LLM 回答：文档中未提及AWE 3.0的全视角通感决策系统（OSD）及其对机器人任务成功率的提升情况。
📎 参考答案：提升300%（即提升3倍）
⭐ 得分：1/5 

──────────────────────────────

### ✏️ 学生反思（15 分）

请在下方回答：

<!-- ✏️ 学生填写 -->

> **上下文记忆在这个任务里为什么有效？它在什么情况下会失效？**
>
> 上下文记忆在这个任务里通过保留和利用历史信息提升模型性能，在信息量适中、结构清晰且与当前任务高度相关时最为有效。在信息过载、冗余或关键信息被截断时会失效。

---
---

# 🔍 第二阶段 — 智能搜索（30 分）

## 背景设置

文档库现在扩大为 **5 篇文章**，10 道评测题的答案分散在不同文章里。
如果把 5 篇全部塞入上下文，token 数量过多且噪声大；
智能搜索的目标是让模型**自己决定去哪篇文章找答案**。

### 预写部分（请勿修改）

以下代码实现了 `search_doc` 工具函数和工具调用框架：

In [8]:
def search_doc(doc_name, query):
    """在指定文章中搜索与query最相关的段落"""
    if doc_name not in docs:
        return f"错误：文章'{doc_name}'不存在。可用文章：{list(docs.keys())}"

    doc_text = docs[doc_name]
    paragraphs = [p.strip() for p in doc_text.split("\n\n") if p.strip()]

    if not paragraphs:
        return "该文章内容为空。"

    query_words = set(query.lower())
    best_para = ""
    best_score = -1

    for para in paragraphs:
        score = sum(1 for w in query.split() if w.lower() in para.lower())
        if score > best_score:
            best_score = score
            best_para = para

    if best_score == 0:
        best_para = paragraphs[0]

    return best_para[:500]

# 文章摘要供模型参考
doc_summaries = {
    "文章A": "Ai2发布MolmoBot和MolmoSpaces，模拟训练零样本迁移至真实机器人",
    "文章B": "ACE Robotics开源Kairos 3.0-4B，具身AI实时世界模型，40亿参数",
    "文章C": "它石智航AWE 3.0具身大模型，工业精密操作，吉尼斯世界纪录",
    "文章D": "理想汽车MindVLA-o1自动驾驶基础模型，MoE Transformer，马赫100芯片",
    "文章E": "蔚来神玑芯片、地平线征程7，中国自动驾驶芯片进展",
}

print("✅ search_doc 工具函数已定义")
print("\n📚 可用文章及摘要：")
for name, summary in doc_summaries.items():
    print(f"   {name}：{summary}")

✅ search_doc 工具函数已定义

📚 可用文章及摘要：
   文章A：Ai2发布MolmoBot和MolmoSpaces，模拟训练零样本迁移至真实机器人
   文章B：ACE Robotics开源Kairos 3.0-4B，具身AI实时世界模型，40亿参数
   文章C：它石智航AWE 3.0具身大模型，工业精密操作，吉尼斯世界纪录
   文章D：理想汽车MindVLA-o1自动驾驶基础模型，MoE Transformer，马赫100芯片
   文章E：蔚来神玑芯片、地平线征程7，中国自动驾驶芯片进展


---
### ✏️ 填空 1 — 定义工具描述（Tool Definition）（10 分）

学生需要填写工具的描述文字，告诉模型这个工具是做什么的、什么时候该用它、参数是什么意思。

**提示：** 工具描述的质量直接影响模型是否能正确决定何时调用工具以及调用哪篇文章。
好的描述应该让模型理解：
1. 这个工具的**用途**是什么
2. **什么时候**应该使用它
3. 每个**参数**代表什么含义

In [9]:
# ✏️ 学生填写：完成工具定义中 [...] 部分

tools = [
    {
        "type": "function",
        "function": {
            "name": "search_doc",
            "description": "用于在指定文档中快速检索与查询最相关的段落，适用于需要从大量文本中精准提取关键信息的问答系统",  # ✏️
            "parameters": {
                "type": "object",
                "properties": {
                    "doc_name": {
                        "description": "目标文章的名称",  # ✏️
                        "type": "string",
                        "enum": ["文章A", "文章B", "文章C", "文章D", "文章E"]
                    },
                    "query": {
                        "description": "查询最相关的段落",  # ✏️
                        "type": "string"
                    }
                },
                "required": ["doc_name", "query"]
            }
        }
    }
]

print("✅ 工具定义已设置")
print(f"   工具描述：{tools[0]['function']['description']}")
print(f"   doc_name 描述：{tools[0]['function']['parameters']['properties']['doc_name']['description']}")
print(f"   query 描述：{tools[0]['function']['parameters']['properties']['query']['description']}")

✅ 工具定义已设置
   工具描述：用于在指定文档中快速检索与查询最相关的段落，适用于需要从大量文本中精准提取关键信息的问答系统
   doc_name 描述：目标文章的名称
   query 描述：查询最相关的段落


---
### ✏️ 填空 2 — 编写工具调用后的处理逻辑（10 分）

模型调用工具后，你需要编写代码：
1. 检查模型是否发出了工具调用
2. 如果是，执行 `search_doc` 函数获取搜索结果
3. 将结果作为 tool 消息追加到 messages
4. 再次调用模型得到最终答案

**提示：**
- `response.choices[0].message.tool_calls` 包含工具调用信息
- 每个 tool_call 有 `.id`、`.function.name`、`.function.arguments`（JSON 字符串）
- tool 消息格式：`{"role": "tool", "tool_call_id": ..., "content": ...}`

In [14]:
def run_agent(question):
    """智能搜索 Agent：让模型自主决定搜索哪篇文章"""

    system_msg = f"""你是一个问答助手。你可以使用 search_doc 工具在文章库中搜索信息来回答问题。
可用文章及其摘要：
{json.dumps(doc_summaries, ensure_ascii=False, indent=2)}

请根据问题内容选择最合适的文章进行搜索，然后基于搜索结果回答问题。"""

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": question}
    ]

    # 第一轮：模型决定是否调用工具
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=tools,
        temperature=0.3,
    )

    assistant_message = response.choices[0].message
    called_doc = "未调用工具"

    # ✅ 学生填写：检查模型是否发出了工具调用，
    # 如果是，执行 search_doc 函数，
    # 并将结果作为 tool 消息追加到 messages，
    # 然后再次调用模型得到最终答案
    #
    # 提示：
    # - assistant_message.tool_calls 是工具调用列表（可能为 None）
    # - tool_call.function.name 是函数名
    # - tool_call.function.arguments 是 JSON 字符串，需要 json.loads() 解析
    # - 调用 search_doc(doc_name, query) 获取搜索结果
    # - 将 assistant_message 和 tool 结果追加到 messages 后再次调用 API

    # [学生填写：约 10-20 行代码]
    if assistant_message.tool_calls:
        tool_call = assistant_message.tool_calls[0]
        if tool_call.function.name == "search_doc":
            args = json.loads(tool_call.function.arguments)
            doc_name = args["doc_name"]
            query = args["query"]
            search_result = search_doc(doc_name, query)
            messages.append({
                "role": "assistant",
                "content": None,
                "tool_calls": [tool_call]
            })
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": search_result
            })
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                temperature=0.3,
            )
            final_answer = response.choices[0].message.content
            called_doc = doc_name
        else:
            final_answer = "错误：不支持的工具调用。"
    else:
        final_answer = assistant_message.content
        called_doc = "未调用工具"

    return final_answer, f"调用了：{called_doc}"

print("✅ run_agent 函数已定义")

✅ run_agent 函数已定义


### 运行第二阶段评测

In [15]:
stage2_score, stage2_results = run_evaluation(
    lambda q: run_agent(q),
    "第二阶段：智能搜索"
)

print(f"\n📊 对比基准提升：{stage2_score - stage1_score:+d} 分")


=== 第二阶段：智能搜索 ===

────────────────────────────────────────────────────────────
📌 问题 1：智元机器人远征A3的满电综合续航最高是多少小时？
🤖 LLM 回答：根据文章A的信息，智元机器人远征A3的满电综合续航最高是12小时。
📎 参考答案：满电综合续航最高8小时
⭐ 得分：2/5 （调用了：文章A）

────────────────────────────────────────────────────────────
📌 问题 2：智元机器人从第5000台到第1万台量产下线大约间隔了多长时间？
🤖 LLM 回答：根据文章A的信息，智元机器人从第5000台到第1万台量产下线大约间隔了半年时间。
📎 参考答案：约一个季度
⭐ 得分：2/5 （调用了：文章A）

────────────────────────────────────────────────────────────
📌 问题 3：Ai2的MolmoSpaces仿真平台包含多少个室内场景和多少个物体模型？
🤖 LLM 回答：Ai2的MolmoSpaces仿真平台包含100个室内场景和1000个物体模型。
📎 参考答案：超过23万个室内场景和13万个物体模型
⭐ 得分：1/5 （调用了：文章A）

────────────────────────────────────────────────────────────
📌 问题 4：ACE Robotics的Kairos 3.0-4B推理速度是Cosmos 2.5的多少倍？能生成最长多久的连贯交互视频？
🤖 LLM 回答：ACE Robotics的Kairos 3.0-4B推理速度是Cosmos 2.5的72倍。它能生成最长1.5倍于视频时长的连贯交互视频，实现实时控制。
📎 参考答案：推理速度是Cosmos 2.5的72倍，能生成最长7分钟的连贯交互视频
⭐ 得分：3/5 （调用了：文章B）

────────────────────────────────────────────────────────────
📌 问题 5：AWE 3.0的全视角通感决策系统（OSD）能将机器人在新视角下的任务成功率提升多少？
🤖 LLM 回答：AWE 3.0的全视角通感决策系统（O

---
---

# 📦 第三阶段 — 检索增强生成 RAG（30 分）

## 背景设置

文档库进一步扩大为**一篇长达 3000 字的技术文档**，已被预切分为 **15 个 chunk**。
RAG 的目标是：提问时只取出最相关的几个 chunk，而非传入整篇文档。

### 预写部分（请勿修改）

以下代码实现了基于 TF-IDF 的检索函数：

In [16]:
# ========================================
# TF-IDF 检索引擎（预实现，请勿修改）
# ========================================

import re
import math
from collections import Counter

def tokenize_chinese(text):
    """简易中文分词（按字+英文词）"""
    tokens = []
    eng_word = ""
    for char in text:
        if char.isascii() and char.isalnum():
            eng_word += char
        else:
            if eng_word:
                tokens.append(eng_word.lower())
                eng_word = ""
            if '\u4e00' <= char <= '\u9fff':
                tokens.append(char)
    if eng_word:
        tokens.append(eng_word.lower())
    return tokens

def compute_tfidf(documents):
    """计算 TF-IDF 向量"""
    n_docs = len(documents)
    doc_tokens = [tokenize_chinese(doc) for doc in documents]

    df = Counter()
    for tokens in doc_tokens:
        for token in set(tokens):
            df[token] += 1

    all_terms = sorted(df.keys())
    term_to_idx = {t: i for i, t in enumerate(all_terms)}

    vectors = []
    for tokens in doc_tokens:
        tf = Counter(tokens)
        vec = np.zeros(len(all_terms))
        for token, count in tf.items():
            if token in term_to_idx:
                idx = term_to_idx[token]
                idf = math.log(n_docs / (1 + df[token]))
                vec[idx] = (count / len(tokens)) * idf
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm
        vectors.append(vec)

    return np.array(vectors), all_terms, term_to_idx

chunk_vectors, vocab, term_to_idx = compute_tfidf(chunks)

def retrieve(query, top_k=3):
    """检索与 query 最相关的 top_k 个 chunk"""
    query_tokens = tokenize_chinese(query)
    query_vec = np.zeros(len(vocab))
    tf = Counter(query_tokens)
    for token, count in tf.items():
        if token in term_to_idx:
            query_vec[term_to_idx[token]] = count / len(query_tokens)

    norm = np.linalg.norm(query_vec)
    if norm > 0:
        query_vec = query_vec / norm

    similarities = chunk_vectors @ query_vec
    top_indices = np.argsort(similarities)[::-1][:top_k]

    return [(int(idx), chunks[idx], float(similarities[idx])) for idx in top_indices]

print("✅ TF-IDF 检索引擎已就绪")
print(f"   词汇表大小：{len(vocab)}")
print(f"   chunk 数量：{len(chunks)}")

# 演示检索效果
demo_results = retrieve("Kairos推理速度", top_k=2)
print(f"\n📎 检索演示（查询：'Kairos推理速度'）：")
for idx, text, sim in demo_results:
    print(f"   chunk_{idx}（相似度 {sim:.3f}）：{text[:60]}...")

✅ TF-IDF 检索引擎已就绪
   词汇表大小：629
   chunk 数量：15

📎 检索演示（查询：'Kairos推理速度'）：
   chunk_6（相似度 0.230）：chunk_6: 4. Kairos 3.0-4B —— 4.1 架构设计
ACE Robotics于2026年3月13...
   chunk_7（相似度 0.141）：chunk_7: 4.2 Kairos 3.0-4B性能与部署能力
Kairos 3.0-4B在多个权威基准测试中排名第...


---
### ✏️ 填空 1 — 决定检索策略（top_k 的选择）（10 分）

你需要选择每次检索返回多少个 chunk（1 到 5 之间），并写出理由。

**思考提示：**
- `top_k` 太小会有什么问题？
- `top_k` 太大会有什么问题？
- 每个 chunk 约 200 字，模型的上下文窗口有限

In [18]:
# ✏️ 学生填写：选择 top_k 的值（1 到 5 之间的整数）

TOP_K = 3

assert TOP_K is not None and 1 <= TOP_K <= 5, "❌ 请将 TOP_K 设置为 1-5 之间的整数"
print(f"✅ TOP_K = {TOP_K}")

✅ TOP_K = 3


<!-- ✏️ 学生填写 -->

> **请写出你选择上面 `TOP_K` 值的理由（10 分）：**
>
> top_k过小可能遗漏关键信息，导致模型无法获取足够上下文生成全面回答；top_k过大可能超出模型有效处理范围；top_k=3相对适中。

---
### ✏️ 填空 2 — 组装 RAG 提示词（10 分）

你需要将检索到的 chunks 组装成发送给模型的提示词。

**要求：**
- 将 `relevant_chunks` 列表中的文本组装成一段上下文
- 清晰告诉模型「以下是相关参考内容，请基于这些内容回答问题」
- 将 `context` 和 `question` 组合成完整的提示词

In [24]:
def build_rag_prompt(question):
    """构建 RAG 提示词"""
    # 检索最相关的 chunks（已实现，直接调用）
    relevant_chunks = retrieve(question, top_k=TOP_K)

    # ✏️ 学生填写：将 relevant_chunks 组装成一段上下文文字
    # relevant_chunks 是一个列表，每个元素是 (chunk_idx, chunk_text, similarity_score)
    # 格式自由，但需要清晰告诉模型："以下是相关参考内容，请基于这些内容回答问题"



    context = "以下是与问题最相关的参考内容：\n"  # ✏️ 学生填写

    for i, (chunk_idx, chunk_text, similarity_score) in enumerate(relevant_chunks, 1):
      context += f"{i}. [相似度: {similarity_score:.2f}] {chunk_text}\n"

    prompt = f"你是一个问答助手。以下是与问题相关的参考内容：\n\n{context}\n请基于以上参考内容回答以下问题：\n{question}\n\n在回答时，请确保：\n1. 答案基于参考内容，不要编造信息。\n2. 如果参考内容不足以回答问题，请说明。\n3. 保持回答简洁明了。"  # ✏️ 学生填写：将 context 和 question 组合成完整提示词

    retrieved_ids = [idx for idx, _, _ in relevant_chunks]
    return prompt, retrieved_ids

print("✅ build_rag_prompt 函数已定义")

✅ build_rag_prompt 函数已定义


In [20]:
def rag_answer(question):
    """第三阶段：RAG 方法"""
    prompt, retrieved_ids = build_rag_prompt(question)

    if not prompt:
        return "请先完成 build_rag_prompt 函数", f"检索到 chunk {retrieved_ids}"

    messages = [
        {"role": "user", "content": prompt}
    ]
    resp = call_llm(messages)
    return resp.choices[0].message.content, f"检索到 chunk {retrieved_ids}"

print("✅ RAG 回答函数已定义")

✅ RAG 回答函数已定义


### 运行第三阶段评测

In [25]:
stage3_score, stage3_results = run_evaluation(
    lambda q: rag_answer(q),
    "第三阶段：RAG"
)


=== 第三阶段：RAG ===

────────────────────────────────────────────────────────────
📌 问题 1：智元机器人远征A3的满电综合续航最高是多少小时？
🤖 LLM 回答：智元机器人远征A3的满电综合续航最高为8小时。
📎 参考答案：满电综合续航最高8小时
⭐ 得分：5/5 （检索到 chunk [2, 3, 1]）

────────────────────────────────────────────────────────────
📌 问题 2：智元机器人从第5000台到第1万台量产下线大约间隔了多长时间？
🤖 LLM 回答：智元机器人从第5000台到第1万台量产下线大约间隔了一个季度。
📎 参考答案：约一个季度
⭐ 得分：5/5 （检索到 chunk [1, 7, 0]）

────────────────────────────────────────────────────────────
📌 问题 3：Ai2的MolmoSpaces仿真平台包含多少个室内场景和多少个物体模型？
🤖 LLM 回答：Ai2的MolmoSpaces仿真平台包含超过23万个室内场景和13万个物体模型。
📎 参考答案：超过23万个室内场景和13万个物体模型
⭐ 得分：5/5 （检索到 chunk [5, 7, 6]）

────────────────────────────────────────────────────────────
📌 问题 4：ACE Robotics的Kairos 3.0-4B推理速度是Cosmos 2.5的多少倍？能生成最长多久的连贯交互视频？
🤖 LLM 回答：ACE Robotics的Kairos 3.0-4B推理速度是Cosmos 2.5的72倍，能生成最长7分钟的连贯交互视频。
📎 参考答案：推理速度是Cosmos 2.5的72倍，能生成最长7分钟的连贯交互视频
⭐ 得分：5/5 （检索到 chunk [7, 6, 0]）

────────────────────────────────────────────────────────────
📌 问题 5：AWE 3.0的全视角通感决策系统（OSD）能将机器人在新视角下的任务成功率提升多少？
🤖 LLM 回答：AWE 

---
---

# 📊 三阶段对比总结

In [23]:
print("\n" + "=" * 50)
print("=== 三阶段对比 ===")
print("=" * 50)
print(f"上下文记忆基准：{stage1_score} / 50")
print(f"智能搜索：      {stage2_score} / 50  ({stage2_score - stage1_score:+d})")
print(f"RAG：           {stage3_score} / 50  ({stage3_score - stage1_score:+d})")
print("=" * 50)


=== 三阶段对比 ===
上下文记忆基准：18 / 50
智能搜索：      35 / 50  (+17)
RAG：           50 / 50  (+32)


---
## 📋 作业提交清单

请确认你已完成以下所有内容：

| 阶段 | 内容 | 完成 |
|------|------|------|
| 第一阶段 | 运行基准测试 | ☐ |
| 第一阶段 | 填写反思回答 | ☐ |
| 第二阶段 | 填写工具描述（3 处） | ☐ |
| 第二阶段 | 编写工具调用处理代码 | ☐ |
| 第二阶段 | 运行评测并查看结果 | ☐ |
| 第三阶段 | 选择 TOP_K 并填写理由 | ☐ |
| 第三阶段 | 编写 RAG 提示词组装代码 | ☐ |
| 第三阶段 | 运行评测并查看对比结果 | ☐ |

**提交方式：** 下载此 notebook（`.ipynb` 文件）并上传到课程平台。